# 01 — Python and Financial Data Fundamentals

**FinRAG learning series · Notebook 1 of 14**

## Learning objectives
By the end of this notebook you will be able to:
1. Use the Python building blocks FinRAG relies on: functions, dataclasses, lists and dictionaries.
2. Explain why financial amounts must be parsed carefully (Indian digit grouping, parentheses for negatives, nil markers) and why FinRAG uses `Decimal` instead of `float`.
3. Load a real CSV and Excel file into **pandas** DataFrames, inspect and clean them.
4. Handle missing values *without inventing data*.
5. Represent financial values in a tidy, traceable structure (value + period + unit + source location).

## Prerequisites
* Basic Python (variables, `if`, `for`).
* The backend virtual environment with `requirements-notebooks.txt` installed, and the Jupyter kernel **"FinRAG (backend venv)"** selected.
* A financial CSV/Excel file in your documents folder (see the setup cell). The outputs saved in this notebook were produced with
  files *transcribed* from a public annual report's standalone statement of profit and loss (see `docs/evaluation/README.md`).

## Why this matters in FinRAG
Every feature in FinRAG — extraction, validation, ratios, RAG answers — ultimately handles **numbers printed in documents**.
A single parsing mistake (reading `(15,124)` as positive, or `-` as zero) silently corrupts every ratio built on it. FinRAG's rule is:
*keep the number exactly as printed, parse it deterministically, and remember where it came from.*

```
document cell/row  ──►  raw text "9,80,136"  ──►  parse_number()  ──►  Decimal('980136')
                                │                                         │
                                └──── kept for display & citation ────────┘  + unit, period, source location
```

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Functions and dataclasses — the shape of FinRAG's data

FinRAG describes *where* a piece of content came from with a small **dataclass** called `Location`
(`backend/app/document_processing/elements.py`). A dataclass is a class whose main job is to hold fields;
Python writes `__init__`, `__repr__` and comparison methods for you.

In [2]:
from app.document_processing.elements import Location, describe_location

pdf_loc = Location(file_format="pdf", page=99, page_label="194")
xlsx_loc = Location(file_format="xlsx", sheet="Statement", cell="B9")
csv_loc = Location(file_format="csv", row=6)

for loc in (pdf_loc, xlsx_loc, csv_loc):
    print(repr(loc.to_dict()), "->", describe_location(loc.to_dict()))

{'file_format': 'pdf', 'page': 99, 'page_label': '194'} -> page 99 (printed p. 194)
{'file_format': 'xlsx', 'sheet': 'Statement', 'cell': 'B9'} -> sheet 'Statement' cell B9
{'file_format': 'csv', 'row': 6} -> row 6


`to_dict()` drops empty fields, and `describe_location()` turns a location into the human-readable text you see in citations.
One `Location` class serves all five formats — the fields that do not apply simply stay `None`.

## 2. Lists and dictionaries
A **list** is an ordered sequence; a **dict** maps keys to values. FinRAG uses dicts for lookups such as
"metric key → definition" and lists for ordered things such as document elements.

In [3]:
from app.financial_analytics.metric_extractor import METRICS, METRIC_NAMES

print("Number of canonical metric definitions:", len(METRICS))
print("First five keys:", [m.key for m in METRICS[:5]])
# dict lookup: key -> human-readable name
for key in ["revenue_from_operations", "net_profit", "total_assets"]:
    print(f"{key:<26} -> {METRIC_NAMES[key]}")

Number of canonical metric definitions: 48
First five keys: ['eps_basic', 'eps_diluted', 'total_equity_and_liabilities', 'total_current_assets', 'total_non_current_assets']
revenue_from_operations    -> Revenue from operations
net_profit                 -> Profit for the period (net profit)
total_assets               -> Total assets


## 3. Why `Decimal`, not `float`
Binary floating point cannot represent most decimal fractions exactly. That is harmless for physics, but unacceptable when a
reconciliation must balance to the rupee/dollar.

In [4]:
from decimal import Decimal

print("float  : 0.1 + 0.2 =", 0.1 + 0.2)
print("Decimal: 0.1 + 0.2 =", Decimal("0.1") + Decimal("0.2"))
# adding ten payments of 0.10 one by one (a running total, as in a ledger loop)
total_f, total_d = 0.0, Decimal("0")
for _ in range(10):
    total_f += 0.1
    total_d += Decimal("0.10")
print("float   running total:", total_f, "| equals 1.0?", total_f == 1.0)
print("Decimal running total:", total_d, "| equals 1.00?", total_d == Decimal("1.00"))
print("float with built-in sum():", sum([0.1] * 10), "(Python 3.12+ sum() uses compensated summation, so it hides the drift)")

float  : 0.1 + 0.2 = 0.30000000000000004
Decimal: 0.1 + 0.2 = 0.3
float   running total: 0.9999999999999999 | equals 1.0? False
Decimal running total: 1.00 | equals 1.00? True
float with built-in sum(): 1.0 (Python 3.12+ sum() uses compensated summation, so it hides the drift)


## 4. Parsing printed financial numbers
Financial documents print numbers in many styles. FinRAG's `parse_number()` (`backend/app/financial_analytics/numbers.py`)
handles the common ones and returns `None` for anything that is not clearly a number. Note especially the **nil markers**:
a dash means "no amount", and FinRAG refuses to turn it into `0` automatically.

In [5]:
from app.financial_analytics.numbers import parse_number

samples = ["9,80,136", "1,234,567", "(15,124)", "-3.5", "51.47", "12.5%", "₹1,000", "-", "—", "nil", "Revenue", "(12"]
for s in samples:
    print(f"{s!r:>12} -> {parse_number(s)!r}")

  '9,80,136' -> Decimal('980136')
 '1,234,567' -> Decimal('1234567')
  '(15,124)' -> Decimal('-15124')
      '-3.5' -> Decimal('-3.5')
     '51.47' -> Decimal('51.47')
     '12.5%' -> Decimal('12.5')
    '₹1,000' -> Decimal('1000')
         '-' -> None
         '—' -> None
       'nil' -> None
   'Revenue' -> None
       '(12' -> None


* `9,80,136` uses **Indian digit grouping** (lakh/crore); `1,234,567` uses Western grouping — both become plain integers.
* `(15,124)` is negative: accountants use parentheses for negatives.
* `-`, `—`, `nil` return `None`: *missing*, not zero.
* `(12` (unbalanced) returns `None` instead of a guess.

## 5. Reading a real CSV with pandas
A **DataFrame** is a table with labelled columns. We read every cell as text (`dtype=str`) so pandas does not reinterpret
`9,80,136` or `(15,124)` on its own — the same choice FinRAG's CSV parser makes.

In [6]:
import pandas as pd

csv_file = nb.doc_path("derived/derived_p58.csv")
df = pd.read_csv(csv_file, dtype=str, keep_default_na=False)
print(df.shape, "rows x columns")
df.head(10)

(28, 3) rows x columns


,Particulars,2024-25,2023-24
0,Value of Sales,"5,38,564","5,33,566"
1,Income from Services,"18,599","41,390"
2,Value of Sales & Services (Revenue),"5,57,163","5,74,956"
3,Less: GST Recovered,"24,371","27,014"
4,Revenue from Operations,"5,32,792","5,47,942"
5,Other Income,"16,094","12,128"
6,Total Income,"5,48,886","5,60,070"
7,Cost of Materials Consumed,"3,84,021","3,76,418"
8,Purchase of Stock-in-Trade,"15,864","13,453"
9,"Changes in Inventories of Finished Goods, Work...","(10,679)","2,700"


### Inspecting structure
`df.columns` are the header row; each other row is a line item. The year columns are text at this point.

In [7]:
print("Columns:", list(df.columns))
print(df.dtypes)

Columns: ['Particulars', '2024-25', '2023-24']
Particulars    str
2024-25        str
2023-24        str
dtype: object


## 6. Cleaning: text → numbers, keeping the original
We add parsed columns **next to** the printed ones instead of overwriting them. That keeps the evidence for every number.

In [8]:
value_cols = [c for c in df.columns if c != "Particulars"]
clean = df.copy()
for c in value_cols:
    clean[c + " (parsed)"] = clean[c].map(lambda v: parse_number(v) if v else None)
clean[["Particulars"] + [c for c in clean.columns if c != "Particulars"]].head(8)

,Particulars,2024-25,2023-24,2024-25 (parsed),2023-24 (parsed)
0,Value of Sales,"5,38,564","5,33,566",538564,533566
1,Income from Services,"18,599","41,390",18599,41390
2,Value of Sales & Services (Revenue),"5,57,163","5,74,956",557163,574956
3,Less: GST Recovered,"24,371","27,014",24371,27014
4,Revenue from Operations,"5,32,792","5,47,942",532792,547942
5,Other Income,"16,094","12,128",16094,12128
6,Total Income,"5,48,886","5,60,070",548886,560070
7,Cost of Materials Consumed,"3,84,021","3,76,418",384021,376418


## 7. Missing values — detect, never invent
`isna()` tells us where parsing produced no number. We report those cells; we do **not** fill them with 0 or with averages.

In [9]:
parsed_cols = [c + " (parsed)" for c in value_cols]
missing = clean[clean[parsed_cols].isna().any(axis=1)][["Particulars"] + value_cols]
print(f"{len(missing)} row(s) with at least one non-numeric or empty value:")
missing

0 row(s) with at least one non-numeric or empty value:


,Particulars,2024-25,2023-24


In this particular file every value cell parsed, so the list is empty — that is a real result, not a placeholder. Try the
same cell on your own spreadsheets: notes columns, `-` for nil and footnote markers are common.

## 8. Reading the same data from Excel
`pd.read_excel` reads one sheet at a time (`sheet_name=None` reads all sheets into a dict). We again keep everything as
objects/text. In notebook 04 you will see how FinRAG reads Excel with **openpyxl** instead, to keep cell coordinates and formulas.

In [10]:
xlsx_file = nb.doc_path("derived/derived_p58.xlsx")
sheets = pd.read_excel(xlsx_file, sheet_name=None, header=None, dtype=object)
for name, frame in sheets.items():
    print(f"sheet {name!r}: {frame.shape[0]} rows x {frame.shape[1]} columns")
sheets["Statement"].head(8)

sheet 'Statement': 32 rows x 3 columns
sheet 'Provenance': 1 rows x 1 columns


,0,1,2
0,Statement of Profit and Loss,NaN,NaN
1,(C in crore),NaN,NaN
2,(C in crore),NaN,NaN
3,Particulars,2024-25,2023-24
4,Value of Sales,"5,38,564","5,33,566"
5,Income from Services,"18,599","41,390"
6,Value of Sales & Services (Revenue),"5,57,163","5,74,956"
7,Less: GST Recovered,"24,371","27,014"


Notice the title rows above the header: `Statement of Profit and Loss` and `(C in crore)`. The unit line says *crore*, but the
currency symbol appears as **"C"** — in the source PDF the ₹ sign is drawn with a private font glyph, and the transcription kept
exactly what the PDF text layer contains. FinRAG therefore does **not** guess the currency from this line (notebook 07 shows how
it is marked *uncertain* instead).

## 9. A tidy, traceable representation
For analysis it is easier to have **one row per value** ("long" format) with its period and its source. This is exactly
the shape of FinRAG's `financial_metrics` table (value + raw value + period + unit + location).

In [11]:
long = df.melt(id_vars="Particulars", var_name="period", value_name="raw_value")
long["value"] = long["raw_value"].map(parse_number)
# source: CSV row number = DataFrame index + 2 (row 1 is the header)
long["source"] = [f"{csv_file.name}, row {i % len(df) + 2}" for i in range(len(long))]
long.head(8)

,Particulars,period,raw_value,value,source
0,Value of Sales,2024-25,"5,38,564",538564,"derived_p58.csv, row 2"
1,Income from Services,2024-25,"18,599",18599,"derived_p58.csv, row 3"
2,Value of Sales & Services (Revenue),2024-25,"5,57,163",557163,"derived_p58.csv, row 4"
3,Less: GST Recovered,2024-25,"24,371",24371,"derived_p58.csv, row 5"
4,Revenue from Operations,2024-25,"5,32,792",532792,"derived_p58.csv, row 6"
5,Other Income,2024-25,"16,094",16094,"derived_p58.csv, row 7"
6,Total Income,2024-25,"5,48,886",548886,"derived_p58.csv, row 8"
7,Cost of Materials Consumed,2024-25,"3,84,021",384021,"derived_p58.csv, row 9"


In [12]:
# Look up one value with its provenance
row = long[(long["Particulars"] == "Revenue from Operations") & (long["period"] == "2024-25")].iloc[0]
print(f"{row['Particulars']} ({row['period']}): printed {row['raw_value']!r} -> {row['value']} | source: {row['source']}")

Revenue from Operations (2024-25): printed '5,32,792' -> 532792 | source: derived_p58.csv, row 6


## Interpretation
* The file was read exactly as printed; numbers were parsed by the same function FinRAG uses in production.
* Rows whose cells are not numbers are *listed*, not repaired.
* The long format carries period and source with every value — the foundation for citations later in the series.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| `FileNotFoundError` from `nb.doc_path` | The document folder is empty | Put your file in `FINRAG_NOTEBOOK_DOCS` or follow `docs/evaluation/README.md` |
| `ModuleNotFoundError: app` | Notebook not started from `notebooks/` or wrong kernel | Start Jupyter from the project and pick "FinRAG (backend venv)" |
| Numbers like `980136.0` appear as floats | pandas converted them | Read with `dtype=str` and parse with `parse_number` |
| Negative amounts become positive | Parentheses ignored | Always parse with `parse_number` |
| `UnicodeDecodeError` reading CSV | Not UTF-8 | Try `encoding="cp1252"` — FinRAG's CSV parser tries several encodings automatically |

## Practical exercises
1. Load one of your own financial CSV files and count how many cells `parse_number` cannot parse. Inspect them — are they nil markers, notes, or genuine errors?
2. Add a `unit` column to the long DataFrame. Where in your document is the unit stated (e.g. "in thousands")?
3. Using `Decimal`, compute total income minus total expenses for one year and compare it with the printed profit-before-tax row.
4. Write a function `is_negative(raw: str) -> bool` and test it on `"(15,124)"`, `"-3.5"`, `"−7"` (Unicode minus).

## Summary
You used the real FinRAG data structures (`Location`, metric definitions) and number parser, loaded real CSV/Excel files with
pandas, cleaned them without losing the printed evidence, refused to invent missing values, and produced a traceable long table.

## Further learning
* Python tutorial — data structures: https://docs.python.org/3/tutorial/datastructures.html
* `decimal` module: https://docs.python.org/3/library/decimal.html
* pandas user guide: https://pandas.pydata.org/docs/user_guide/index.html
* Next: **02 — PDF processing**.